# INC-04: Detecting the B+ tree's full scan

The first plot and the detection table use the original B+ tree on cimy-mbp. Selectivity is the fraction of distinct tempo keys returned. Counters are divided by the number of queries. Timing uses the median and quartiles of batch-average query time over five replicates.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
if root.name == "analysis":
    root = root.parent

timings = pd.read_csv(root / "results" / "timings.csv")
baseline = timings.loc[
    timings["in_docker"].eq(True) & timings["machine_id"].eq("cimy-mbp")
].copy()
baseline["us_per_query"] = baseline["total_ns"] / baseline["n_ops"] / 1_000
baseline["keys_per_query"] = baseline["comparisons"] / baseline["n_ops"]
baseline["nodes_per_query"] = baseline["nodes_visited"] / baseline["n_ops"]

e3 = baseline.loc[
    baseline["experiment"].eq("E3_range_tempo")
    & baseline["code_version"].eq("5a59833")
    & baseline["session_id"].eq("3d1109855014")
]
e4 = baseline.loc[
    baseline["experiment"].eq("E4_prefix_track_name")
    & baseline["code_version"].eq("751ca9a")
    & baseline["session_id"].eq("27c2b42c5b4f")
]

In [ ]:
indexes = {"bplus_tree": "B+ tree (order 32)", "avl_tree": "AVL tree", "sorted_list": "Sorted list"}
grouped = (
    e3.loc[e3["index_type"].isin(indexes)]
    .groupby(["index_type", "operation_param"], as_index=False)
    .agg(keys_per_query=("keys_per_query", "median"))
)
styles = {
    "bplus_tree": {"color": "C0", "marker": "o"},
    "avl_tree": {"color": "C1", "linestyle": "--", "marker": ".", "zorder": 3},
    "sorted_list": {"color": "C2", "marker": "s", "markersize": 4},
}
fig, ax = plt.subplots(figsize=(6.5, 3.8), layout="constrained")
for name, label in indexes.items():
    rows = grouped.loc[grouped["index_type"].eq(name)]
    ax.loglog(100 * rows["operation_param"], rows["keys_per_query"], label=label, **styles[name])
ax.set(
    title="Original B+ tree examines every tempo key",
    xlabel="Range selectivity (% of distinct tempo keys)",
    ylabel="Keys examined per query",
)
ax.grid(axis="y", alpha=0.25)
ax.legend()
output = root / "analysis" / "figures_INC04"
output.mkdir(exist_ok=True)
fig.savefig(output / "INC-04_range_detection.png", dpi=180)
plt.show()

In [ ]:
selected = pd.concat([
    e3.loc[e3["operation_param"].isin([0.0001, 0.01])],
    e4.loc[e4["operation_param"].eq(3)],
])
detection = (
    selected.loc[selected["index_type"].eq("bplus_tree") & selected["index_param"].eq("order=32")]
    .groupby(["experiment", "code_version", "operation_param"], as_index=False)
    .agg(
        reps=("rep", "nunique"),
        height=("height", "median"),
        median_us=("us_per_query", "median"),
        q1_us=("us_per_query", lambda x: x.quantile(0.25)),
        q3_us=("us_per_query", lambda x: x.quantile(0.75)),
        keys_per_query=("keys_per_query", "median"),
        nodes_per_query=("nodes_per_query", "median"),
    )
)
display(detection.round({"median_us": 2, "q1_us": 2, "q3_us": 2, "keys_per_query": 2, "nodes_per_query": 2}))